In [70]:
import pandas as pd, numpy as np, random, os, math
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader,SequentialSampler,RandomSampler,TensorDataset
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
!pip install sacrebleu
from sacrebleu.metrics import BLEU
import unicodedata
import string
import re

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cpu


In [71]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:

csv_file_path = '/content/drive/MyDrive/english_french.csv'
df = pd.read_csv(csv_file_path)
print(df.columns.tolist(), len(df))

df.columns = ['English', 'French']

df = df.sample(30_000, random_state=SEED).reset_index(drop=True)
print(f"Using {len(df)} pairs")
print(df.head(3))

['English', 'French'] 229803
Using 30000 pairs
                                     English  \
0       If you give me a book, I'll read it.   
1  We'll leave tomorrow, weather permitting.   
2     I'm so happy, I feel like I could fly.   

                                            French  
0         Si vous me donnez un livre, je le lirai.  
1    Nous partirons demain, si le temps le permet.  
2  Je suis si heureux que je m'élève dans le ciel.  


In [73]:
MAX_LENGTH = 30

PAD_token = 0
SOS_token = 1
EOS_token = 2
UNK_token = 3
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)



This code is the data preprocessing part for an English → French translation model. Its job is to take raw sentences, clean them, turn words into numbers, pad them, and create a PyTorch DataLoader so the model can train in batches.

# This class stores the vocabulary for one language.

##What it is used for

It keeps track of:

every word seen in the dataset


the numeric index of each word


how many times each word appears

That is needed because neural networks do not understand text directly. They need numbers.

In [74]:
class Lang:
  def __init__(self,name):
    self.name=name
    self.word2index = {} # word → number
    self.word2count = {} # word → frequency
    self.index2word = { # number → word
    PAD_token: "PAD", # used to fill short sentences so all sentences have the same length
    SOS_token: "SOS", # start of sentence
    EOS_token: "EOS", # end of sentence
    UNK_token: "UNK"} # unknown word
    self.n_words = 4 # vocabulary starts with 4 special tokens

  # It splits a sentence into words and adds each word to the vocabulary,Then each word is stored.
  # This builds the vocabulary from  dataset.
  def addSentence(self,sentence):
    for word in sentence.split(' '):
      self.addWord(word)


  # If the word is new, assign it a new index,If it already exists, increase its count
  #This is how the vocabulary grows word by word.
  def addWord(self,word):
    if word not in self.word2index:
      self.word2index[word] = self.n_words
      self.word2count[word] = 1
      self.index2word[self.n_words] = word
      self.n_words += 1
    else:
      self.word2count[word] += 1

  def trim(self, min_count):
    keep_words = []
    for k, v in self.word2count.items():
      if v >= min_count:
        keep_words.append(k)



#It checks whether a pair of sentences is acceptable.
# This removes long or unsuitable sentence pairs.
def filterPair(pair):
  return len(pair[0].split(' ')) < MAX_LENGTH and \
    len(pair[1].split(' ')) < MAX_LENGTH and\
    pair[1].startswith(eng_prefixes)



#It applies filterPair() to all sentence pairs.
#This gives you a cleaned dataset.
def filterPairs(pairs):
  return [pair for pair in pairs if filterPair(pair)]



# It removes accents from letters.
# This makes text more uniform and easier to process.
def unicodeToAscii(s):
  return''.join(
      c for c in unicodedata.normalize('NFD', s)
      if unicodedata.category(c) != 'Mn'
  )


# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
  s = unicodeToAscii(s.lower().strip())
  s=re.sub(r"([.!?])", r" \1", s)
  s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
  return s


# It creates sentence pairs from your DataFrame.
def readLangsFromDF(df, reverse=False):

  pairs = [
      [
          normalizeString(row['English']),
          normalizeString(row['French'])
      ]
      for _, row in df.iterrows()
  ]

  if reverse:
      pairs = [list(reversed(p)) for p in pairs]
      input_lang = Lang("fra")
      output_lang = Lang("eng")
  else:
      input_lang = Lang("eng")
      output_lang = Lang("fra")

  return input_lang, output_lang, pairs

#What it does
    #This is the full preprocessing pipeline:
    #read the pairs from the DataFrame
    #filter the pairs
    #count all words
    #build vocabularies
    #This is the main function that prepares everything before training.
def prepareDataFromDF(df, reverse=False):

  input_lang, output_lang, pairs = readLangsFromDF(df, reverse)

  print("Read %s sentence pairs" % len(pairs))

  pairs = filterPairs(pairs)

  print("Trimmed to %s sentence pairs" % len(pairs))

  print("Counting words...")

  for pair in pairs:
      input_lang.addSentence(pair[0])
      output_lang.addSentence(pair[1])

  print("Counted words:")
  print(input_lang.name, input_lang.n_words)
  print(output_lang.name, output_lang.n_words)
  return input_lang, output_lang, pairs

#Convert a sentence into a list of word indexes.
def indexFromSentence(lang,sentence):
  return[lang.word2index.get(word, UNK_token) for word in sentence.split(' ')]+[EOS_token]

# It converts the list of indexes into a PyTorch tensor.
def tensorFromSentence(lang,sentence):
  indexes=indexFromSentence(lang,sentence)
  return torch.tensor(indexes,dtype=torch.long,device=device).view(-1,1)


#It converts one sentence pair into two tensors:
#input tensor
#target tensor
def tensorFromPair(pair,input_lang,output_lang):
  input_tensor=tensorFromSentence(input_lang,pair[0])
  target_tensor=tensorFromSentence(output_lang,pair[1])
  return(input_tensor,target_tensor)


#prepare the data
#convert each sentence to indexes
#pad sentences to MAX_LENGTH
#create masks
#build a TensorDataset
#create a DataLoader

def get_dataloader(batch_size):
  # Pass the global df here
  input_lang, output_lang, pairs = prepareDataFromDF(df, True)

  n = len(pairs)
  input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)#The number form of the input sentence.
  input_mask = np.zeros((n, MAX_LENGTH), dtype=np.int32)# Marks real words with 1 and padding wit
  target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32) # The number form of the output sentence.
  target_mask = np.zeros((n, MAX_LENGTH), dtype=np.int32) # Same idea for target sentences.

  for idx, (inp, tgt) in enumerate(pairs):
      inp_ids = indexFromSentence(input_lang, inp)
      tgt_ids = indexFromSentence(output_lang, tgt)
      input_ids[idx, :len(inp_ids)] = inp_ids
      input_mask[idx, :len(inp_ids)] = 1
      target_ids[idx, :len(tgt_ids)] = tgt_ids
      target_mask[idx, :len(tgt_ids)] = 1

  train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                             torch.LongTensor(input_mask).to(device),
                             torch.LongTensor(target_ids).to(device),
                             torch.LongTensor(target_mask).to(device))

  train_sampler = RandomSampler(train_data)
  train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
  return input_lang, output_lang, train_dataloader

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W1 = nn.Linear(hidden_size, hidden_size)#transforms the query
        self.W2 = nn.Linear(hidden_size, hidden_size)#transforms the values
        self.V = nn.Linear(hidden_size, 1)#V turns the combined result into a single attention score
    def forward(self, query, values, mask):
        """
        query: decoder state at the current step, shape [B, 1, H]
        values: encoder outputs, shape [B, M, H]
        mask: tells which encoder positions are valid, shape [B, M]
        query:  [B, 1, H]
        values: [B, M, H]
        mask:   [B, M]
        """
        query_expanded = query.expand(-1, values.size(1), -1)  # [B, M, H]
        scores = self.V(torch.tanh(self.W1(query_expanded) + self.W2(values)))  # [B, M, 1]
        scores = scores.transpose(1, 2)  # [B, 1, M]

        scores=scores.masked_fill(mask.unsqueeze(1) == 0, -1e9)
        alphas = F.softmax(scores, dim=-1)  # [B, 1, M]

        context = torch.bmm(alphas, values)  # [B, 1, H]
        return context, alphas

In [75]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, input_seq, hidden):
        """
        input_seq: [B, T]
        hidden:    [1, B, H]
        """
        embedded = self.embedding(input_seq)   # [B, T, H]
        output, hidden = self.gru(embedded, hidden)
        return output, hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(1, batch_size, self.hidden_size, device=device)

In [76]:
class AttentionDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.attention = BahdanauAttention(hidden_size)

        self.concat = nn.Linear(hidden_size * 2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.bridge = nn.Linear(hidden_size, hidden_size)

    def forward(self, encoder_outputs, encoder_hidden, input_mask,
                target_tensor=None, device="cpu"):
        """
        encoder_outputs: [B, M, H]
        encoder_hidden:  [1, B, H]
        input_mask:      [B, M]
        target_tensor:   [B, T] or None
        """
        batch_size = encoder_outputs.size(0)

        decoder_input = torch.full(
            (batch_size, 1),
            SOS_token,
            dtype=torch.long,
            device=device
        )

        decoder_hidden = torch.tanh(self.bridge(encoder_hidden))  # [1, B, H]

        decoder_outputs = []
        attentions = []

        steps = target_tensor.size(1) if target_tensor is not None else MAX_LENGTH

        for t in range(steps):
            decoder_output, decoder_hidden, attn = self.forward_step(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                input_mask
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn)

            if target_tensor is not None:
                decoder_input = target_tensor[:, t].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1, dim=-1)  # [B, 1, 1]
                decoder_input = topi.squeeze(-1).detach()  # [B, 1]

        decoder_outputs = torch.cat(decoder_outputs, dim=1)  # [B, T, V]
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        attentions = torch.cat(attentions, dim=1)  # [B, T, M]
        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input_token, hidden, encoder_outputs, input_mask):
        """
        input_token:     [B, 1]
        hidden:          [1, B, H]
        encoder_outputs: [B, M, H]
        input_mask:      [B, M]
        """
        embedded = self.embedding(input_token)   # [B, 1, H]
        embedded = F.relu(embedded)

        gru_output, hidden = self.gru(embedded, hidden)  # [B, 1, H], [1, B, H]

        query = gru_output  # [B, 1, H]
        context, attn = self.attention(query, encoder_outputs, input_mask)  # [B, 1, H], [B, 1, M]

        combined = torch.cat((gru_output, context), dim=2)  # [B, 1, 2H]
        combined = torch.tanh(self.concat(combined))        # [B, 1, H]

        output = self.out(combined)  # [B, 1, V]
        return output, hidden, attn

In [78]:
class EncoderDecoder(nn.Module):
    def __init__(self, hidden_size, input_vocab_size, output_vocab_size):
        super(EncoderDecoder, self).__init__()
        self.encoder = EncoderRNN(input_vocab_size, hidden_size)
        self.decoder = AttentionDecoderRNN(hidden_size, output_vocab_size)
        # self.decoder = DecoderRNN(hidden_size, output_vocab_size)

    def forward(self, inputs, input_mask, targets=None):
        batch_size = inputs.size(0)
        initial_hidden = self.encoder.init_hidden(batch_size, inputs.device)
        encoder_outputs, encoder_hidden = self.encoder(inputs, initial_hidden)
        decoder_outputs, decoder_hidden, attentions = self.decoder(
            encoder_outputs, encoder_hidden, input_mask, targets)
        return decoder_outputs, decoder_hidden

In [79]:
PAD_idx = 0
SOS_token = 1
EOS_token = 2
hidden_size = 256
batch_size = 32

def train(train_dataloader, model, n_epochs, learning_rate=0.0003):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss(ignore_index=PAD_idx)

    for epoch in range(1, n_epochs + 1):
        loss = 0
        for iter, batch in enumerate(train_dataloader):
            # Batch tensors: [B, SeqLen]
            input_tensor  = batch[0]
            input_mask    = batch[1]
            target_tensor = batch[2]
            loss += train_step(input_tensor, input_mask, target_tensor,
                               model, optimizer, criterion)
        print('Epoch {} Loss {}'.format(epoch, loss / iter))


def train_step(input_tensor, input_mask, target_tensor, model,
               optimizer, criterion):
    optimizer.zero_grad()
    decoder_outputs, decoder_hidden = model(input_tensor, input_mask, target_tensor, device=input_tensor.device)

    # Collapse [B, Seq] dimensions for NLL Loss
    loss = criterion(
        decoder_outputs.view(-1, decoder_outputs.size(-1)), # [B, Seq, OutVoc] -> [B*Seq, OutVoc]
        target_tensor.view(-1) # [B, Seq] -> [B*Seq]
    )

    loss.backward()
    optimizer.step()
    return loss.item()

def ids2words(lang, ids):
    return [lang.index2word[idx] for idx in ids]

def greedy_decode(model, dataloader, input_lang, output_lang):
    with torch.no_grad():
        batch = next(iter(dataloader))
        input_tensor  = batch[0]
        input_mask    = batch[1]
        target_tensor = batch[2]

        # Call the model for inference. EncoderDecoder will use MAX_LENGTH for decoding.
        decoder_outputs, _ = model(input_tensor, input_mask, device=input_tensor.device)
        # We only need the top predicted token for each step
        _, topi = decoder_outputs.topk(1, dim=-1) # topi is [B, MAX_LENGTH, 1]
        decoded_ids = topi.squeeze(-1) # decoded_ids is [B, MAX_LENGTH]

        for idx in range(input_tensor.size(0)):
            input_sent_ids = input_tensor[idx].cpu().numpy()
            target_sent_ids = target_tensor[idx].cpu().numpy()
            output_sent_ids = decoded_ids[idx].cpu().numpy()

            # Convert input sentence IDs to words, stopping at EOS or PAD
            input_words = []
            for token_id in input_sent_ids:
                if token_id == EOS_token or token_id == PAD_token:
                    break
                input_words.append(input_lang.index2word[token_id])

            # Convert target sentence IDs to words, stopping at EOS or PAD
            target_words = []
            for token_id in target_sent_ids:
                if token_id == EOS_token or token_id == PAD_token:
                    break
                target_words.append(output_lang.index2word[token_id])

            # Convert output sentence IDs to words, stopping at EOS or PAD
            output_words = []
            for token_id in output_sent_ids:
                if token_id == EOS_token or token_id == PAD_token: # Stop at EOS or PAD
                    break
                # Handle UNK_token display
                if token_id == UNK_token:
                    output_words.append('<UNK>')
                else:
                    output_words.append(output_lang.index2word[token_id])


            print('Input:  {}'.format(' '.join(input_words)))
            print('Target: {}'.format(' '.join(target_words)))
            print('Output: {}'.format(' '.join(output_words)))

In [80]:
class EncoderDecoder(nn.Module):
    def __init__(self, hidden_size, input_vocab_size, output_vocab_size):
        super(EncoderDecoder, self).__init__()
        self.encoder = EncoderRNN(input_vocab_size, hidden_size)
        self.decoder = AttentionDecoderRNN(hidden_size, output_vocab_size)
        # self.decoder = DecoderRNN(hidden_size, output_vocab_size)

    def forward(self, inputs, input_mask, targets=None, device="cpu"):
        batch_size = inputs.size(0)
        initial_hidden = self.encoder.init_hidden(batch_size, inputs.device)
        encoder_outputs, encoder_hidden = self.encoder(inputs, initial_hidden)
        decoder_outputs, decoder_hidden, attentions = self.decoder(
            encoder_outputs, encoder_hidden, input_mask, targets, device=device)
        return decoder_outputs, decoder_hidden

In [81]:
    input_lang, output_lang, train_dataloader =  get_dataloader(batch_size)
    model_instance = EncoderDecoder(hidden_size, input_lang.n_words, output_lang.n_words).to(device)
    train(train_dataloader, model_instance, n_epochs=20)
    greedy_decode(model_instance, train_dataloader, input_lang, output_lang)

Read 30000 sentence pairs
Trimmed to 2719 sentence pairs
Counting words...
Counted words:
fra 2302
eng 1793
Epoch 1 Loss 4.838421469643002
Epoch 2 Loss 3.5963813265164695
Epoch 3 Loss 3.203510974134718
Epoch 4 Loss 2.9174241565522694
Epoch 5 Loss 2.6885348842257546
Epoch 6 Loss 2.4974304948534285
Epoch 7 Loss 2.327046658311571
Epoch 8 Loss 2.1728519314811345
Epoch 9 Loss 2.033522286585399
Epoch 10 Loss 1.9040515167372567
Epoch 11 Loss 1.7843600440592993
Epoch 12 Loss 1.6744638312430609
Epoch 13 Loss 1.5724158173515683
Epoch 14 Loss 1.467498562165669
Epoch 15 Loss 1.3776047925154369
Epoch 16 Loss 1.2878854828221458
Epoch 17 Loss 1.1935526508660543
Epoch 18 Loss 1.110288876862753
Epoch 19 Loss 1.0266278414499193
Epoch 20 Loss 0.9490574059032258
Input:  je ne suis vraiment pas occupee .
Target: i m really not busy .
Output: i m really not busy .
Input:  il n a jamais ete amoureux auparavant .
Target: he s never been in love before .
Output: he is never to be at home .
Input:  je suis deso